In [1]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/pune_property_clean.csv")

X = df.drop(columns=["price"])
y = np.log(df["price"])          # we predict log(price), as the EDA showed

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
print(Xtr.shape, Xte.shape)

(22741, 50) (5686, 50)


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

cat_cols = ["facing", "locality", "neworold", "ownership", "status", "property_type"]
num_cols = [c for c in X.columns if c not in cat_cols]

prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_cols),
])

model = Pipeline([
    ("prep", prep),
    ("rf", RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)),
])
model.fit(Xtr, ytr)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](50,)","['age','area','balconies',...,'am_golf_course','am_wifi','amenity_count']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,50
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concat

In [3]:
from sklearn.metrics import mean_absolute_error, r2_score

pred = np.exp(model.predict(Xte))
actual = np.exp(yte)

print("R²                  :", round(r2_score(actual, pred), 3))
print("MAE (₹ lakh)        :", round(mean_absolute_error(actual, pred) / 1e5, 1))
print("Median % error      :", round((abs(pred - actual) / actual).median() * 100, 1), "%")

R²                  : 0.948
MAE (₹ lakh)        : 18.4
Median % error      : 8.4 %


In [4]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.base import clone

models = {
    "Random Forest": RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8,
                            n_jobs=-1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=31,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, verbose=-1),
}

rows = []
for name, m in models.items():
    pipe = Pipeline([("prep", clone(prep)), ("model", m)])
    pipe.fit(Xtr, ytr)
    pred = np.exp(pipe.predict(Xte))
    actual = np.exp(yte)
    rows.append({
        "model": name,
        "R2": round(r2_score(actual, pred), 3),
        "MAE_lakh": round(mean_absolute_error(actual, pred) / 1e5, 1),
        "median_pct_error": round((abs(pred - actual) / actual).median() * 100, 1),
    })

results = pd.DataFrame(rows)
results

,model,R2,MAE_lakh,median_pct_error
0,Random Forest,0.948,18.4,8.4
1,XGBoost,0.946,19.6,9.6
2,LightGBM,0.947,19.4,9.4


In [5]:
import joblib, os

final_model = Pipeline([
    ("prep", clone(prep)),
    ("rf", RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                 n_jobs=-1, random_state=42)),
])
final_model.fit(Xtr, ytr)

pred = np.exp(final_model.predict(Xte))
actual = np.exp(yte)
print("R²           :", round(r2_score(actual, pred), 3))
print("MAE (₹ lakh) :", round(mean_absolute_error(actual, pred) / 1e5, 1))
print("Median % err :", round((abs(pred - actual) / actual).median() * 100, 1), "%")

R²           : 0.944
MAE (₹ lakh) : 19.3
Median % err : 9.0 %


In [6]:
joblib.dump(final_model, "../models/rf_price_model.joblib", compress=3)
joblib.dump(list(X.columns), "../models/feature_columns.joblib")
print("model size (MB):", round(os.path.getsize("../models/rf_price_model.joblib") / 1e6, 1))

df["price_per_sqft"] = df["price"] / df["area"]
stats = (df.groupby("locality")
           .agg(listings=("price", "size"), median_psf=("price_per_sqft", "median"))
           .reset_index())
stats.to_csv("../data/processed/locality_stats.csv", index=False)
print(stats.shape)

model size (MB): 17.4
(419, 3)


In [7]:
trn_psf = np.exp(ytr) / Xtr["area"]
pune_psf = trn_psf.median()

g = trn_psf.groupby(Xtr["locality"]).agg(["median", "size"])
k = 10   # small localities are pulled toward the Pune average
g["locality_psf"] = (g["median"] * g["size"] + pune_psf * k) / (g["size"] + k)
loc_map = g["locality_psf"].to_dict()

def add_locality_psf(X):
    X = X.copy()
    X["locality_psf"] = X["locality"].map(loc_map).fillna(pune_psf)
    return X

Xtr2, Xte2 = add_locality_psf(Xtr), add_locality_psf(Xte)
print(loc_map.get("deccan gymkhana"), pune_psf)

11734.02384209499 6976.744186046517


In [8]:
num_cols2 = num_cols + ["locality_psf"]
prep2 = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), num_cols2),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat_cols),
])
final_model2 = Pipeline([
    ("prep", prep2),
    ("rf", RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                 n_jobs=-1, random_state=42)),
])
final_model2.fit(Xtr2, ytr)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](51,)","['age','area','balconies',...,'am_wifi','amenity_count','locality_psf']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,51
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concaten

In [9]:
actual = np.exp(yte)
counts = Xtr["locality"].value_counts()
rare = (Xte["locality"].map(counts).fillna(0) < 20).values

def score(name, pred_log):
    pred = np.exp(pred_log)
    err = (abs(pred - actual) / actual)
    print(f"{name:12s} R²={r2_score(actual, pred):.3f} | "
          f"MAE={mean_absolute_error(actual, pred)/1e5:.1f} L | "
          f"median err all={err.median()*100:.1f}% | "
          f"rare localities={err[rare].median()*100:.1f}%")

score("Old model", final_model.predict(Xte))
score("New model", final_model2.predict(Xte2))
print("test rows in rare localities:", rare.sum())

Old model    R²=0.944 | MAE=19.3 L | median err all=9.0% | rare localities=18.9%
New model    R²=0.954 | MAE=17.8 L | median err all=8.2% | rare localities=13.2%
test rows in rare localities: 278


In [10]:
joblib.dump(final_model2, "../models/rf_price_model.joblib", compress=3)
joblib.dump(list(Xtr2.columns), "../models/feature_columns.joblib")
joblib.dump({"map": loc_map, "default": float(pune_psf)}, "../models/locality_psf.joblib")
print("model size (MB):", round(os.path.getsize("../models/rf_price_model.joblib") / 1e6, 1))

model size (MB): 17.4
